# Polygon area thresholding

for the methods section

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    # Font
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # Lines and markers
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Axes
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    # Layout
    "figure.constrained_layout.use": True,
})


In [ ]:
import geopandas as gpd
import utca
import osmnx as ox
import numpy as np
from skimage.filters import threshold_otsu

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    # Font
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # Lines and markers
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Axes
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    # Layout
    "figure.constrained_layout.use": True,
})

cm = 1 / 2.54

In [ ]:
varos = "Kecskemét"
path_str = f'output/neat_20260103_171612/{varos}_simplified.graphml'
G = ox.load_graphml(path_str)
G = utca.prepare_graph(G)
poly_sequence = utca.polygonize(G)
polygons = utca.poly_df(G, poly_sequence)

In [ ]:
utca.explore_graph(G, polygons)

In [ ]:
import matplotlib.pyplot as plt
val = 'area'

# Get the histogram data
#data = np.log1p(polygons[val])
data = polygons[val]
hist_counts, hist_bins = np.histogram(data, bins=50)
bin_centers = (hist_bins[:-1] + hist_bins[1:]) / 2

# Calculate Otsu threshold
#threshold = threshold_otsu(hist=(hist_counts, hist_bins[1:]))
threshold = threshold_otsu(hist=(hist_counts, bin_centers))
#threshold = threshold_otsu(data.to_numpy())
quantile_threshold = data.quantile(0.99)

# Plot the histogram
fig, ax = plt.subplots(figsize=(8*cm, 7*cm))
ax.hist(data, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(threshold, color='green', linestyle='--', linewidth=1, label=f'Otsu Threshold')#: {threshold:.2f}')
ax.axvline(quantile_threshold, color='red', linestyle='--', linewidth=1, label=f'Quantile Threshold')#: {threshold:.2f}')
plt.xlabel('Area')
plt.ylabel('Frequency')
#plt.title('Histogram of Polygon Areas with Otsu Threshold')
plt.semilogy()
plt.legend()
#plt.grid(True)#, alpha=0.3)
plt.show()

print(f"Otsu threshold value: {(threshold):.2f}")

In [ ]:
polygons['threshold'] = polygons['area'] > quantile_threshold

In [ ]:
utca.explore_graph(G, polygons, poly_column_to_plot='threshold')

In [ ]:
def plot_map(varos, save=False):
    #center = shapely.Point(admin_centres[admin_centres['name'] == varos].iloc[0][['lon', 'lat']].values)
    path_str = f'output/neat_20260103_171612/{varos}_simplified.graphml'
    G = ox.load_graphml(path_str)
    G = utca.prepare_graph(G)
    polygons = utca.poly_df(G)
    val = 'area'

    data = polygons[val]
    #data = np.log1p(polygons[val])
    #hist_counts, hist_bins = np.histogram(data, bins=50)
    #bin_centers = (hist_bins[:-1] + hist_bins[1:]) / 2
    #threshold = threshold_otsu(hist=(hist_counts, bin_centers))
    #threshold = data.quantile(0.99)
    threshold = 100000
    
    polygons['threshold'] = polygons['area'] > threshold

    fig, ax = plt.subplots(figsize=(8*cm, 7*cm))
    polygons.plot(ax=ax, column='threshold', legend=True, legend_kwds={'title': 'Above threshold', 'loc': 3})
    ax.axis("off")
    if save:
        fig.savefig("output/figs_maj10/polygon_map.pdf")

In [ ]:
plot_map("Cegléd")

In [ ]:
def plot_hist(varos, save=False):
    path_str = f'output/neat_20260103_171612/{varos}_simplified.graphml'
    G = ox.load_graphml(path_str)
    G = utca.prepare_graph(G)
    polygons = utca.poly_df(G)
    val = 'area'

    # Get the histogram data
    #data = np.log1p(polygons[val])
    data = polygons[val]
    hist_counts, hist_bins = np.histogram(data, bins=50)
    bin_centers = (hist_bins[:-1] + hist_bins[1:]) / 2

    # Calculate Otsu threshold
    #threshold = threshold_otsu(hist=(hist_counts, hist_bins[1:]))
    threshold = threshold_otsu(hist=(hist_counts, bin_centers))
    #threshold = threshold_otsu(data.to_numpy())
    quantile_threshold = data.quantile(0.99)
    const_threshold = 100000

    # Plot the histogram
    fig, ax = plt.subplots(figsize=(8*cm, 7*cm))
    ax.hist(data, bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(threshold, color='green', linestyle='--', linewidth=1, label=f'Otsu Threshold')#: {threshold:.2f}')
    ax.axvline(quantile_threshold, color='0.3', linestyle='--', linewidth=1, label=f'Quantile Threshold')#: {threshold:.2f}')
    ax.axvline(const_threshold, color='red', linestyle='--', linewidth=1, label=f'Constant Threshold')#: {threshold:.2f}')
    plt.xlabel('Area')
    plt.ylabel('Frequency')
    #plt.title('Histogram of Polygon Areas with Otsu Threshold')
    plt.semilogy()
    plt.legend()
    #plt.grid(True)#, alpha=0.3)
    plt.show()
    if save:
        fig.savefig("output/figs_maj10/poly_hist.pdf")

In [ ]:
plot_hist("Cegléd")